# 📰 CheckIt.AI - Fake News Killer
## Projet P12 - Formation IA : Extrayez des données multimodales

---

### 📋 Présentation du projet

**Objectif :** Industrialiser l'acquisition de données multimodales (texte + images) depuis le web pour alimenter un détecteur de fake news.

**Problématique :** Comment automatiser la collecte, le nettoyage et la classification de contenus d'actualité provenance de sources multiples, tout en distinguant les opinions controversées de la désinformation ?

---

### 🎯 Les 5 Livrables de ce projet

| # | Livrable | Description |
|---|----------|-------------|
| 1 | **Rapport d'exploration** | Analyse des sources multimodales disponibles |
| 2 | **Scripts d'extraction** | APIs, RSS, Scraping (BeautifulSoup, Selenium, Playwright) |
| 3 | **Pipeline ETL + MCD** | Transformation et Modèle Conceptuel de Données |
| 4 | **DAG Airflow** | Orchestration automatisée du cycle ETL |
| 5 | **Dashboard Streamlit** | Monitoring et KPIs en temps réel |

---

### 🏗️ Architecture générale du système

```
┌─────────────┐     ┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│  EXTRACTION │ ──► │TRANSFORMATION│ ──► │  STOCKAGE   │ ──► │ MONITORING  │
│             │     │             │     │             │     │             │
│ • RSS       │     │ • Nettoyage │     │ • SQLite    │     │ • Streamlit │
│ • APIs      │     │ • Normalis. │     │ • Fichiers  │     │ • KPIs      │
│ • Scraping  │     │ • Classif.  │     │             │     │ • Alertes   │
└─────────────┘     └─────────────┘     └─────────────┘     └─────────────┘
```

---

### 📚 Sommaire du notebook

1. **Mission 1 : Sourcing & Exploration** - Identification des sources multimodales
2. **Mission 2 : Ingénierie d'Extraction** - Scripts de scraping et APIs
3. **Mission 3 : Transformation & Modélisation** - Pipeline ETL et MCD
4. **Mission 4 : Orchestration Airflow** - DAG automatisé
5. **Mission 5 : Monitoring** - Dashboard Streamlit

---

# MISSION 1 : Sourcing & Exploration

---

## 1.1 Objectif de la mission

Identifier et évaluer les sources multimodales (texte + images) disponibles sur le web pour alimenter notre détecteur de fake news.

### 🔍 Types de sources explorées :

- **APIs REST** : Interfaces standardisées avec JSON
- **Flux RSS** : syndication de contenu actualités
- **Web Scraping** : extraction directe depuis les pages HTML

### ⚖️ Distinction éthique : Opinion vs Désinformation

| Critère | Opinion controversée | Désinformation |
|---------|----------------------|----------------|
| **Source** | Médias identifiés, éditorialistes | Sources non identifiées |
| **Intentionnalité** | Point de vue assumé | Tromperie délibérée |
| **Vérifiabilité** | Faits vérifiables | Fausses données |
| **Traitement** | À analyser | À exclure/rejeter |

## 1.2 Sources APIs évaluées

### 📡 NewsAPI (newsapi.org)

```
Type: API REST
Gratuit: 100 requêtes/jour
Données: Titres, descriptions, URL source, date
Images: ✅ Oui (URL vers image)
Catégories: business, entertainment, general, health, science, sports, technology
```

**Avantages :** Structure JSON claire, images disponibles
**Inconvénients :** Limite quotidienne, nécessitent inscription

---

### 📡 Reddit API

```
Type: API REST + Scraping
Gratuit: ✅ Illimité (avec authentification OAuth)
Données: Posts, commentaires, métadonnées
Images: ✅ Oui (upload ou URL)
Subreddits: r/news, r/worldnews, r/politics
Rate limit: 60 requêtes/minute
```

**Avantages :** Contenu varié, multimodal, gratuit
**Inconvénients :** Contenu utilisateur, qualité variable

---

### 📡 HackerNews API

```
Type: API REST Firebase
Gratuit: ✅ Illimité, sans clé
Données: Stories, comments, scores
Images: ❌ Non
Endpoint: https://hacker-news.firebaseio.com/v0/
```

**Avantages :** Gratuit, sans authentification, qualité technique
**Inconvénients :** Pas d'images, contenu technique

## 1.3 Sources RSS sélectionnées

Les flux RSS offrent un accès gratuit et illimité aux actualités.

| Source | URL RSS | Type de contenu |
|--------|---------|-----------------|
| **Le Figaro** | `https://www.lefigaro.fr/rss/figaro_actualites.xml` | Actualités générales |
| **Le Monde** | `https://www.lemonde.fr/rss/une.xml` | Actualités générales |
| **20 Minutes** | `https://www.20minutes.fr/feeds/rss-une.xml` | Actualités grand public |
| **Nouvel Obs** | `https://www.nouvelobs.com/rss.xml` | Société, culture |
| **Euronews** | `https://www.euronews.com/rss` | International |
| **Breitbart** | `https://feeds.feedburner.com/breitbart` | Actualités US |

### Pourquoi RSS ?

1. **Gratuit et illimité** - Pas de quotas ni de clés API
2. **Format standardisé** - XML structuré, facile à parser
3. **Multimodalité** - Inclut souvent les images via `<enclosure>` ou `<media:content>`
4. **Légitime** - Contenu publié par des médias reconnus

## 1.4 Conclusion Mission 1 - Livrable 1

### ✅ Rapport d'exploration des sources

**Fichier :** `docs/exploration_sources.md`

### Technologies identifiées pour l'extraction :

```python
import requests           # API calls HTTP
from bs4 import BeautifulSoup  # HTML parsing
import feedparser        # RSS parsing alternatif
import urllib            # Image download
```

### Plan d'action validé :

1. **Implémenter un script Python avec `requests`** pour les APIs
2. **Fallback : scraper les flux RSS** (Le Figaro, Le Monde)
3. **Extraire les images** avec `urllib` ou `requests`
4. **Stocker localement** en JSON/CSV

---

### 🎯 Points clés à retenir :

- Plusieurs sources complémentaires (RSS + APIs + Scraping)
- La solution combinée offre la meilleure résilience
- Distinction éthique Opinion vs Désinformation dès la sélection

---

# MISSION 2 : Ingénierie d'Extraction (Scraping & API)

---

## 2.1 Objectif de la mission

Développer des scripts modulaires d'extraction capables de récupérer des données depuis différentes sources :

- **Flux RSS** : via BeautifulSoup
- **APIs REST** : Reddit, HackerNews, NewsAPI
- **Web Scraping** : BeautifulSoup, Selenium, Playwright

### Architecture des scripts :

```
src/extraction/
├── main.py              # Orchestrateur CLI
├── rss_fetcher.py      # Extraction RSS
├── api_client.py        # APIs (Reddit, HackerNews, NewsAPI)
├── scraper_bs4.py       # BeautifulSoup
├── scraper_selenium.py  # Selenium (JS)
└── scraper_playwright.py # Playwright (JS)

## 2.2 Extraction RSS avec BeautifulSoup

### Principe de fonctionnement

Le protocole RSS (Really Simple Syndication) permet de récupérer des flux d'actualités au format XML structuré.

### Code详解 - `rss_fetcher.py`

```python
import requests
from bs4 import BeautifulSoup
from datetime import datetime

class RSSFetcher:
    """Gestionnaire de flux RSS."""
    
    # Sources RSS configurées
    DEFAULT_FEEDS = {
        "le_figaro": "https://www.lefigaro.fr/rss/figaro_actualites.xml",
        "le_monde": "https://www.lemonde.fr/rss/une.xml",
        "nouvel_obs": "https://www.nouvelobs.com/rss.xml",
        "20_minutes": "https://www.20minutes.fr/feeds/rss-une.xml",
        "euronews": "https://www.euronews.com/rss",
        "breitbart": "https://feeds.feedburner.com/breitbart",
    }
    
    def fetch_all(self, limit_per_feed: int = 10) -> list[dict]:
        """Récupère tous les flux RSS."""
        all_articles = []
        for name, url in self.feeds.items():
            articles = self.fetch_feed(name, url, limit_per_feed)
            all_articles.extend(articles)
        return all_articles
```

### Points clés :

1. **Session persistante** : Réutilise les connexions TCP
2. **User-Agent personnalisé** : Identifie notre application
3. **Timeout** : Évite les blocages (15 secondes)
4. **Gestion d'erreurs** : Try/Except pour chaque requête

## 2.3 Extraction des images multimodales

### Problématique

Les flux RSS peuvent contenir des images de plusieurs façons :

1. **`<enclosure>`** : Balise standard pour les pièces jointes
2. **`<media:content>`** : Extension pour les médias
3. **`<media:thumbnail>`** : Miniature de l'article
4. **Dans le `<description>`** : Image incluse dans le HTML

### Code de gestion multimodale

```python
def _parse_item(self, feed_name: str, item) -> Optional[dict]:
    """Parse un élément RSS."""
    # Extraction de l'image depuis enclosure
    image_url = ""
    enclosure = item.find("enclosure")
    if enclosure and enclosure.get("type", "").startswith("image"):
        image_url = enclosure.get("url", "")
    
    # Fallback: media:content ou media:thumbnail
    if not image_url:
        media = item.find("media:content") or item.find("media:thumbnail")
        if media:
            image_url = media.get("url", "")
    
    # Fallback: extraction depuis description HTML
    if not image_url and description:
        img_soup = BeautifulSoup(description.get_text("", strip=True), "html.parser")
        img_tag = img_soup.find("img")
        if img_tag:
            image_url = img_tag.get("src", "")
```

### Format de données unifié

Chaque article est normalisé avec la structure suivante :

```python
{
    "title": "Titre de l'article",
    "description": "Résumé de l'article",
    "content": "Contenu complet",
    "url": "https://exemple.com/article",
    "image_url": "https://exemple.com/image.jpg",
    "source": "le_figaro",
    "author": "",
    "published_at": "2026-04-17T10:30:00",
    "extracted_at": "2026-04-17T12:00:00",
    "type": "rss"
}
```

## 2.4 Extraction via APIs REST

### Client HackerNews (sans clé)

HackerNews offre une API gratuite et ouverte via Firebase.

```python
class HackerNewsClient:
    """Client pour l'API Hacker News (gratuit, sans clé)."""
    BASE_URL = "https://hacker-news.firebaseio.com/v0"
    
    def get_top_stories(self, limit: int = 20) -> list[dict]:
        """Récupère les meilleures histoires."""
        # 1. Récupérer la liste des IDs
        response = requests.get(f"{self.BASE_URL}/topstories.json")
        story_ids = response.json()[:limit]
        
        # 2. Pour chaque ID, récupérer les détails
        articles = []
        for story_id in story_ids:
            item_response = requests.get(
                f"{self.BASE_URL}/item/{story_id}.json"
            )
            item = item_response.json()
            # Normalisation au format standard...
        return articles
```

---

### Client Reddit API

```python
class RedditClient:
    """Client pour l'API Reddit."""
    BASE_URL = "https://www.reddit.com"
    
    def get_hot_posts(self, subreddit: str = "news", limit: int = 25) -> list[dict]:
        """Récupère les posts populaires d'un subreddit."""
        endpoint = f"{self.BASE_URL}/r/{subreddit}/hot.json"
        params = {"limit": limit}
        
        response = requests.get(
            endpoint, 
            headers={"User-Agent": "CheckIt.AI/1.0"},
            params=params
        )
        data = response.json()
        posts = data.get("data", {}).get("children", [])
        return self._normalize_posts(posts)
```

### Gestion des erreurs et logs

```python
import logging

logger = logging.getLogger(__name__)

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()  # Soulève une exception si HTTP 4xx/5xx
    data = response.json()
    logger.info(f"Récupéré {len(articles)} articles")
    
except requests.RequestException as e:
    logger.error(f"Erreur requête: {e}")
    return []
```

## 2.5 Web Scraping avec BeautifulSoup

### Quand utiliser BeautifulSoup ?

- Sites statiques (HTML généré côté serveur)
- Structure HTML cohérente
- Pas de contenu chargé en JavaScript

### Code示例

```python
from bs4 import BeautifulSoup
import requests

class NewsScraper:
    """Scraper pour les sites d'actualité."""
    
    USER_AGENT = "CheckIt.AI/1.0 (Research Project)"
    
    def scrape_site(self, url: str, selectors: dict, limit: int = 10) -> list[dict]:
        """Scrape un site avec les sélecteurs CSS fournis."""
        response = self.session.get(url, timeout=15)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, "html.parser")
        articles = []
        
        # Extraction selon les sélecteurs CSS
        items = soup.select(selectors.get("article", "article"))[:limit]
        
        for item in items:
            article = self._extract_article(item, selectors)
            if article:
                articles.append(article)
        
        return articles
    
    def _extract_article(self, element, selectors: dict) -> Optional[dict]:
        """Extrait les données d'un élément article."""
        title_el = element.select_one(selectors.get("title", "h2, h3"))
        desc_el = element.select_one(selectors.get("description", "p"))
        link_el = element.select_one(selectors.get("link", "a"))
        img_el = element.select_one(selectors.get("image", "img"))
        
        return {
            "title": title_el.get_text(strip=True),
            "description": desc_el.get_text(strip=True)[:500] if desc_el else "",
            "url": link_el.get("href", "") if link_el else "",
            "image_url": img_el.get("src", "") if img_el else "",
        }
```

### Sélecteurs CSS pour différents sites

```python
SITE_SELECTORS = {
    "le_figaro": {
        "url": "https://www.lefigaro.fr/",
        "article": "article, .article-card",
        "title": "h1, h2",
        "description": "p",
        "link": "a",
        "image": "img",
    },
    "20_minutes": {
        "url": "https://www.20minutes.fr/",
        "article": "article, .article",
        "title": "h2, h3",
        "description": ".article-chapo",
    }
}
```

## 2.6 Web Scraping avec Selenium

### Quand utiliser Selenium ?

- Sites dynamiques (JavaScript charge le contenu)
- Contenu rendu côté client (SPA)
- Interactions nécessaires (clics, scrolls)

### Architecture Selenium

```
┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│   Python    │────►│   Chrome    │────►│   Website    │
│   Script    │     │   Driver    │     │   (Browser)  │
└─────────────┘     └─────────────┘     └─────────────┘
                           │
                           ▼
                    ┌─────────────┐
                    │   HTML DOM   │
                    │  (rendered)  │
                    └─────────────┘
```

### Code Selenium - Configuration

```python
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

class SeleniumScraper:
    """Scraper utilisant Selenium pour les sites JavaScript."""
    
    def _init_driver(self):
        """Initialise le driver Selenium."""
        options = Options()
        options.add_argument("--headless")  # Mode sans interface graphique
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--disable-gpu")
        options.add_argument("user-agent=CheckIt.AI/1.0")
        
        self.driver = webdriver.Chrome(options=options)
        return True
```

### Code Selenium - Extraction

```python
def scrape_dynamic_site(self, url: str, article_selector: str = "article", limit: int = 10) -> list[dict]:
    """Scrape un site nécessitant JavaScript."""
    if not self._init_driver():
        return []
    
    self.driver.get(url)
    
    # Attendre que les articles chargent (attente explicite)
    WebDriverWait(self.driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, article_selector))
    )
    
    # Extraire tous les éléments article
    articles_elements = self.driver.find_elements(
        By.CSS_SELECTOR, article_selector
    )[:limit]
    
    articles = []
    for elem in articles_elements:
        article = self._extract_dynamic_article(elem)
        if article:
            articles.append(article)
    
    self.driver.quit()  # Toujours fermer le navigateur
    return articles
```

### Points importants :

1. **Headless mode** : Exécution sans interface graphique (serveur)
2. **WebDriverWait** : Évite les conditions de course
3. **Always quit()** : Libère les ressources dans le finally

## 2.7 Web Scraping avec Playwright

### Pourquoi Playwright ?

- Plus moderne que Selenium
- Meilleure performance
 API Python native
- Support natif des screenshots et PDFs

### Code Playwright

```python
from playwright.sync_api import sync_playwright
import time

def extract_playwright(url: str = "https://www.20minutes.fr/", limit: int = 10):
    """Extraction via Playwright."""
    articles = []
    
    with sync_playwright() as p:
        # Lancer Chromium en mode headless
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        
        # Navigation avec timeout
        page.goto(url, timeout=15000)
        time.sleep(2)  # Attendre le rendu JS
        
        # Sélection des éléments
        articles_elem = page.query_selector_all("article")
        
        for elem in articles_elem[:limit]:
            try:
                title_elem = elem.query_selector("h2, h3, .title")
                link_elem = elem.query_selector("a")
                
                title = title_elem.inner_text() if title_elem else ""
                link = link_elem.get_attribute("href") if link_elem else ""
                
                if title and link:
                    articles.append({
                        "title": title.strip(),
                        "url": link,
                        "source": "20_minutes",
                        "type": "scraped_playwright"
                    })
            except:
                continue
        
        browser.close()
    
    return articles
```

### Comparatif des outils de scraping

| Critère | BeautifulSoup | Selenium | Playwright |
|---------|---------------|----------|------------|
| **Vitesse** | ⭐⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐⭐ |
| **JavaScript** | ❌ | ✅ | ✅ |
| **Simplicité** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ |
| **Ressources** | Faibles | Élevées | Modérées |
| **Maintenance** | Facile | Moyenne | Facile |

## 2.8 Orchestrateur CLI - main.py

### Principe

Un point d'entrée unique pour lancer les extractions depuis la ligne de commande.

```python
# src/extraction/main.py

import argparse
import logging
from pathlib import Path
from datetime import datetime

from src.extraction.rss_fetcher import RSSFetcher
from src.extraction.api_client import (
    RedditClient,
    HackerNewsClient,
    NewsAPIAIClient
)
from src.extraction.scraper_bs4 import NewsScraper

def main():
    """CLI pour l'extraction de données."""
    parser = argparse.ArgumentParser(description="CheckIt.AI - Extraction")
    parser.add_argument("--source", choices=["rss", "reddit", "hn", "all"], default="all")
    parser.add_argument("--output", default="data/raw")
    parser.add_argument("--limit", type=int, default=10)
    
    args = parser.parse_args()
    
    all_articles = []
    
    # Extraction selon la source
    if args.source in ["rss", "all"]:
        fetcher = RSSFetcher()
        articles = fetcher.fetch_all(limit_per_feed=args.limit)
        all_articles.extend(articles)
    
    if args.source in ["reddit", "all"]:
        reddit = RedditClient()
        posts = reddit.get_hot_posts("news", limit=args.limit)
        all_articles.extend(posts)
    
    # Sauvegarde
    output_file = f"{args.output}/extract_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(all_articles, f, ensure_ascii=False, indent=2)
    
    print(f"✓ {len(all_articles)} articles extraits -> {output_file}")

if __name__ == "__main__":
    main()
```

### Utilisation

```bash
# Extraction RSS
pixi run python src/extraction/main.py --source rss

# Extraction Reddit uniquement
pixi run python src/extraction/main.py --source reddit --limit 20

# Toutes les sources
pixi run python src/extraction/main.py --source all
```

## 2.9 Conclusion Mission 2 - Livrable 2

### ✅ Scripts d'extraction automatisée

**Fichiers produits :**

| Fichier | Description | Source |
|---------|-------------|--------|
| `rss_fetcher.py` | Extraction flux RSS | 6 médias FR/US |
| `api_client.py` | APIs REST | Reddit, HackerNews, NewsAPI |
| `scraper_bs4.py` | Scraping BeautifulSoup | Sites statiques |
| `scraper_selenium.py` | Scraping Selenium | Sites JavaScript |
| `scraper_playwright.py` | Scraping Playwright | Sites modernes |
| `main.py` | Orchestrateur CLI | Point d'entrée |

---

### 🔑 Points clés de l'extraction :

1. **Format unifié** : Tous les scripts sortent le même JSON
2. **Gestion d'erreurs** : Try/Except + logging sur chaque requête
3. **Rate limiting** : Respect des quotas APIs
4. **User-Agent** : Identification correcte auprès des serveurs
5. **Multimodalité** : Extraction images via enclosure/media

---

# MISSION 3 : Transformation & Modélisation (ETL)

---

## 3.1 Objectif de la mission

Transformer les données brutes extraites en données nettoyées, normalisées et classifiées.

### Pipeline ETL

```
EXTRACTION          TRANSFORMATION           STOCKAGE
──────────────────────────────────────────────────────────
│                  │                        │
▼                  ▼                        ▼
Données brutes ──► Nettoyage ──► Normalisation ──► Base SQLite
                  │                        │
                  ▼                        ▼
              Validation              Classification
              (titre ?)              (Opinion vs Disinfo)
                  │                        │
                  ▼                        ▼
              Enrichissement             KPIs
              (keywords,                (métriques
               multimodal)                de qualité)
```

## 3.2 Nettoyage des données - DataCleaner

### Problématique

Les données brutes contiennent souvent :

- Balises HTML résiduelles
- Caractères spéciaux
- Espaces multiples
- URLs intégrées au texte
- Données nulles ou vides

### Code de nettoyage

```python
import re

class DataCleaner:
    """Nettoyage des données extraites."""
    
    def clean(self, article: dict) -> Optional[dict]:
        """Nettoie un article."""
        cleaned = article.copy()
        
        # Titre obligatoire
        title = cleaned.get("title", "")
        if not title or title == "[Removed]":
            return None
        
        cleaned["title"] = self._clean_text(title)
        cleaned["description"] = self._clean_text(cleaned.get("description", ""))[:500]
        cleaned["content"] = self._clean_text(cleaned.get("content", ""))
        
        return cleaned
    
    def _clean_text(self, text: str) -> str:
        """Nettoie le texte."""
        if not text:
            return ""
        
        # Normaliser les espaces
        text = re.sub(r'\s+', ' ', text)
        
        # Supprimer les URLs
        text = re.sub(r'https?://\S+', '', text)
        
        # Supprimer les balises HTML
        text = re.sub(r'<[^>]+>', '', text)
        
        return text.strip()
```

### Opérations de nettoyage

| Étape | Expression | Example avant → après |
|-------|------------|------------------------|
| Espaces | `\s+` → ` ` | `"Paris   est  belle"` → `"Paris est belle"` |
| URLs | `https?://\S+` → `""` | `"Découvrez https://t.co/abc"` → `"Découvrez"` |
| HTML | `<[^>]+>` → `""` | `"<b>Paris</b>"` → `"Paris"` |

## 3.3 Normalisation du texte - TextNormalizer

### Pourquoi normaliser ?

La normalisation prépare le texte pour :

1. **Extraction de mots-clés** : Facilite l'identification des termes importants
2. **Classification** : Rend le texte comparable
3. **Recherche** : Permet les correspondances

### Code de normalisation

```python
class TextNormalizer:
    """Normalisation du texte pour analyse."""
    
    STOP_WORDS = {
        "le", "la", "les", "un", "une", "des", "de", "du", "au",
        "et", "ou", "mais", "donc", "car", "ni", "que", "qui",
        "est", "sont", "été", "a", "ont", "avec", "pour", "par"
    }
    
    @staticmethod
    def normalize(text: str) -> str:
        """Normalise le texte."""
        if not text:
            return ""
        
        # Minuscules
        text = text.lower()
        
        # Supprimer la ponctuation
        text = re.sub(r'[^\w\s]', ' ', text)
        
        # Normaliser les espaces
        text = re.sub(r'\s+', ' ', text)
        
        return text.strip()
    
    @staticmethod
    def extract_keywords(text: str, max_keywords: int = 10) -> list[str]:
        """Extrait les mots-clés significatifs."""
        if not text:
            return []
        
        words = text.lower().split()
        
        # Filtrer les stop words et mots courts
        keywords = [
            w for w in words 
            if len(w) > 3 and w not in TextNormalizer.STOP_WORDS
        ]
        
        # Compter les occurrences
        from collections import Counter
        counter = Counter(keywords)
        
        return [w for w, _ in counter.most_common(max_keywords)]
```

### Exemple de résultat

```
Texte original:
"Le gouvernement français a annoncé aujourd'hui une nouvelle réforme"

Texte normalisé:
"gouvernement français annoncé aujourdhui nouvelle réforme"

Mots-clés extraits:
['gouvernement', 'français', 'annoncé', 'aujourdhui', 'nouvelle', 'réforme']
```

## 3.4 Validation des données - DataValidator

### Règles de validation

Un article est considéré comme **valide** si :

1. **Titre obligatoire** : Non vide et différent de [Removed]
2. **Contenu minimum** : Au moins une description ou du contenu
3. **Multimodalité** : Vérification de la présence d'image

### Code de validation

```python
class DataValidator:
    """Validation des données."""
    
    @staticmethod
    def is_valid_article(article: dict) -> bool:
        """Vérifie si l'article est valide."""
        if not article:
            return False
        
        # Titre obligatoire
        if not article.get("title"):
            return False
        
        # Au moins une source de contenu
        if not any([
            article.get("description"),
            article.get("content")
        ]):
            return False
        
        return True
    
    @staticmethod
    def is_multimodal(article: dict) -> bool:
        """Vérifie si l'article est multimodal (avec image)."""
        return bool(article.get("image_url"))
    
    @staticmethod
    def validate_batch(articles: list[dict]) -> dict:
        """Valide un lot et retourne des statistiques."""
        total = len(articles)
        valid = sum(1 for a in articles if DataValidator.is_valid_article(a))
        multimodal = sum(1 for a in articles if DataValidator.is_multimodal(a))
        
        return {
            "total": total,
            "valid": valid,
            "invalid": total - valid,
            "multimodal": multimodal,
            "text_only": total - multimodal,
            "success_rate": valid / max(total, 1) * 100
        }
```

### Statistiques de validation

```
┌────────────────────────────────────────────┐
│  VALIDATION BATCH                          │
├────────────────────────────────────────────┤
│  Total articles:    100                     │
│  Valides:           85 ✓                   │
│  Multimodaux:       72 (avec image)        │
│  Texte seul:        13                     │
│  Rejetés:           15 ✗                   │
│                                            │
│  Taux de succès:   85%                     │
└────────────────────────────────────────────┘
```

## 3.5 Classification Opinion vs Désinformation

### Approche méthodologique

Notre classifier utilise une approche **basée sur des règles (Rule-Based)** avec des indicateurs sémantiques.

### Indicateurs d'OPINION controversée

| Catégorie | Indicateurs |
|---------|--------------|
| **Sources connues** | Le Figaro, Le Monde, AFP, Reuters, BBC |
| **Verbes d'attribution** | selon, déclare, estime, explique |
| **Types d'articles** | éditorial, chronique, tribune, opinion |

```python
OPINION_INDICATORS = {
    "positive": [
        "selon", "d'après", "estime", "considère", "affirme",
        "déclare", "explique", "éditorial", "chronique",
        "tribune", "opinion", "point de vue"
    ],
    "sources_connues": [
        "le figaro", "le monde", "le point", "20 minutes",
        "france info", "reuters", "afp", "bbc"
    ]
}
```

### Indicateurs de DÉSINFORMATION

| Catégorie | Indicateurs |
|---------|--------------|
| **Alertes** | urgent, breaking, exclusif, scandale |
| **Émotions** | haine, colère, peur, panique, fake |
| **Sources douteuses** | sources anonymes, on apprend que... |
| **Absence de vérification** | n'a pas pu être vérifié, à confirmer |

## 3.6 Code du Classifier

```python
class ContentClassifier:
    """Classifieur de contenu Opinion vs Désinformation."""
    
    def classify(self, article: dict) -> dict:
        """Classifie un article."""
        title = article.get("title", "").lower()
        description = article.get("description", "").lower()
        content = article.get("content", "").lower()
        source = article.get("source", "").lower()
        
        text = f"{title} {description} {content}"
        
        # Calcul des scores
        opinion_score = self._calculate_opinion_score(text, source)
        disinfo_score = self._calculate_disinfo_score(text, source)
        
        # Détermination de la catégorie
        if opinion_score > 0.3 and disinfo_score < 0.4:
            category = "opinion"
            confidence = min(opinion_score + 0.2, 1.0)
        elif disinfo_score > 0.5:
            category = "desinformation"
            confidence = min(disinfo_score, 1.0)
        elif disinfo_score > 0.3 or opinion_score > 0.2:
            category = "neutre"
            confidence = 0.5
        else:
            category = "indetermine"
            confidence = 0.3
        
        return {
            "category": category,
            "confidence": round(confidence, 2),
            "opinion_score": round(opinion_score, 2),
            "disinfo_score": round(disinfo_score, 2),
        }
    
    def _calculate_opinion_score(self, text: str, source: str) -> float:
        """Calcule le score d'opinion."""
        score = 0.0
        factors = 0
        
        # Source connue
        for known_source in OPINION_INDICATORS["sources_connues"]:
            if known_source in source:
                score += 0.3
                factors += 1
                break
        
        # Indicateurs d'opinion
        for indicator in OPINION_INDICATORS["positive"]:
            if indicator in text:
                score += 0.15
                factors += 1
        
        return min(score, 1.0) if factors > 0 else 0.0
```

### Exemple de classification

```
Article 1: "Le gouvernement annonce une nouvelle réforme"
- Source: le_monde
- Score Opinion: 0.45
- Score Disinfo: 0.0
- Résultat: OPINION (confiance: 0.65)

Article 2: "SCANDALE: Le gouvernement cache la vérité!"
- Source: inconnu.xyz
- Score Opinion: 0.0
- Score Disinfo: 0.65
- Résultat: DÉSINFORMATION (confiance: 0.65)
```

## 3.7 Pipeline ETL complet - pipeline.py

```python
class TransformationPipeline:
    """Pipeline ETL complet."""
    
    def transform(self, articles: list[dict]) -> list[dict]:
        """Transforme les articles."""
        logger.info(f"Transformation de {len(articles)} articles...")
        
        # 1. NETTOYAGE
        cleaned = self.cleaner.clean_batch(articles)
        
        # 2. NORMALISATION DU TEXTE
        for article in cleaned:
            article["normalized_title"] = TextNormalizer.normalize(
                article.get("title", "")
            )
            article["keywords"] = TextNormalizer.extract_keywords(
                article.get("title", "") + " " + article.get("description", "")
            )
        
        # 3. VALIDATION
        valid_articles = [
            a for a in cleaned 
            if self.validator.is_valid_article(a)
        ]
        
        # 4. ENRICHISSEMENT
        for article in valid_articles:
            article["is_multimodal"] = self.validator.is_multimodal(article)
            article["word_count"] = len(
                (article.get("title", "") + " " + article.get("description", "")).split()
            )
        
        # 5. CLASSIFICATION
        classified = self.classifier.classify_batch(valid_articles)
        
        # 6. PERSISTANCE EN BASE
        if self.db:
            count = self.db.insert_articles_batch(classified)
            logger.info(f"Articles insérés en DB: {count}")
        
        return classified
```

### Flux de transformation

```
Données brutes (100 articles)
         │
         ▼
┌─────────────────┐
│ 1. NETTOYAGE    │ ──► 95 articles (5 rejetés)
└─────────────────┘
         │
         ▼
┌─────────────────┐
│ 2. NORMALISATION │ ──► normalized_title, keywords
└─────────────────┘
         │
         ▼
┌─────────────────┐
│ 3. VALIDATION   │ ──► 90 articles valides
└─────────────────┘
         │
         ▼
┌─────────────────┐
│ 4. ENRICHISSEMT │ ──► is_multimodal, word_count
└─────────────────┘
         │
         ▼
┌─────────────────┐
│ 5. CLASSIFICAT. │ ──► opinion/desinformation/neutre
└─────────────────┘
         │
         ▼
┌─────────────────┐
│ 6. STOCKAGE     │ ──► SQLite + JSON/CSV
└─────────────────┘
```

## 3.8 Modèle Conceptuel de Données (MCD)

### Schéma relationnel en Mermaid

```mermaid
erDiagram
    ARTICLE ||--o{ KEYWORD : has
    ARTICLE }|..|{ SOURCE : comes_from
    ARTICLE {
        string id PK
        string title
        text description
        text content
        string url
        string image_url
        string source FK
        string author
        datetime published_at
        datetime extracted_at
        string type
        string normalized_title
        bool is_multimodal
        int word_count
    }
    
    SOURCE {
        string id PK
        string name
        string type
        string url
        datetime last_fetch
    }
    
    KEYWORD {
        string id PK
        string article_id FK
        string word
        int frequency
    }
    
    EXTRACTION_LOG {
        int id PK
        string source
        datetime start_time
        datetime end_time
        int articles_count
        int errors_count
        string status
    }
    
    METRICS {
        int id PK
        datetime timestamp
        string metric_name
        float value
        string unit
    }
```

### Explication des entités

| Entité | Description | Relations |
|--------|-------------|-----------|
| **ARTICLE** | Article d'actualité extrait | PK: id, FK: source |
| **SOURCE** | Source de données (RSS, API, Scraping) | 1:N avec ARTICLE |
| **KEYWORD** | Mot-clé extrait de l'article | N:1 avec ARTICLE |
| **EXTRACTION_LOG** | Journal d'extraction | Traçabilité |
| **METRICS** | Métriques de monitoring | Suivi des KPIs |

## 3.9 Conclusion Mission 3 - Livrable 3

### ✅ Pipeline de transformation et schéma de données

**Fichiers produits :**

| Fichier | Description |
|---------|-------------|
| `pipeline.py` | Pipeline ETL complet |
| `cleaner.py` | Nettoyage et normalisation |
| `classifier.py` | Classification Opinion vs Désinformation |
| `database.py` | Base SQLite |
| `schema.mmd` | MCD Mermaid |

---

### 🔑 Points clés de la transformation :

1. **Nettoyage** : Suppression HTML, URLs, normalisation
2. **Validation** : Titre obligatoire, contenu minimum
3. **Enrichissement** : Multimodalité, nombre de mots, keywords
4. **Classification** : Score Opinion vs Désinformation
5. **MCD** : Modèle relationnel pour la persistance

---

# MISSION 4 : Orchestration avec Apache Airflow

---

## 4.1 Objectif de la mission

Automatiser le cycle ETL complet (Extraction → Transformation → Chargement) avec Apache Airflow.

### Qu'est-ce qu'Apache Airflow ?

Airflow est un **orchestrateur de workflows** qui permet de :

- Définir des **DAGs** (Directed Acyclic Graphs)
- Planifier des **tâches** (schedule)
- Gérer les **dépendances** entre tâches
- Surveiller l'**exécution**
- Relancer les **échecs** automatiquement

### Architecture Airflow

```
┌─────────────────────────────────────────────────────┐
│                  Airflow                           │
│  ┌─────────────┐  ┌─────────────┐  ┌───────────┐ │
│  │  Scheduler  │  │   Webserver │  │  Worker   │ │
│  │  (planifie) │  │  (interface) │  │ (exécute) │ │
│  └──────┬──────┘  └──────┬──────┘  └─────┬─────┘ │
│         │                │               │       │
│         └────────────────┼───────────────┘       │
│                          │                        │
│                    ┌──────┴──────┐                 │
│                    │  Metadata   │                 │
│                    │   Database  │                 │
│                    │  (SQLite/PG) │                 │
│                    └─────────────┘                 │
└─────────────────────────────────────────────────────┘
```

## 4.2 Structure du DAG - fakenews_pipeline.py

### Configuration du DAG

```python
from datetime import datetime, timedelta
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator
from airflow.sdk import TaskGroup

DEFAULT_ARGS = {
    "owner": "CheckIt.AI",
    "depends_on_past": False,
    "email_on_failure": False,
    "retries": 2,                    # 2 tentatives
    "retry_delay": timedelta(minutes=5),  # 5 min entre retries
}

with DAG(
    dag_id="fakenews_etl_pipeline",
    default_args=DEFAULT_ARGS,
    description="Pipeline ETL pour l'extraction de données multimodales",
    schedule="0 */2 * * *",         # Toutes les 2 heures
    start_date=datetime(2026, 4, 1),
    catchup=False,
    tags=["fakenews", "etl", "multimodal"]
) as dag:
    pass
```

### Paramètres clés

| Paramètre | Valeur | Description |
|-----------|--------|-------------|
| **schedule** | `0 */2 * * *` | Toutes les 2 heures |
| **start_date** | 2026-04-01 | Date de début |
| **retries** | 2 | Nombre de tentatives |
| **retry_delay** | 5 minutes | Délai entre tentatives |
| **catchup** | False | Ne pas rattraper le passé |

## 4.3 Tâches d'extraction dans le DAG

### Tâche RSS

```python
def extract_rss(ti):
    """Tâche d'extraction RSS."""
    from src.extraction.rss_fetcher import RSSFetcher
    
    fetcher = RSSFetcher()
    articles = fetcher.fetch_all(limit_per_feed=15)
    
    # Sauvegarder dans un fichier
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filepath = f"{DATA_RAW}/rss_extract_{timestamp}.json"
    
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(articles, f, ensure_ascii=False)
    
    # Passer le chemin aux tâches suivantes via XCom
    ti.xcom_push(key="rss_file", value=filepath)
    ti.xcom_push(key="rss_count", value=len(articles))
    
    return filepath
```

### Tâches parallèles (TaskGroup)

```python
with TaskGroup("extraction_group") as extraction:
    # Toutes ces tâches s'exécutent en PARALLÈLE
    extract_rss = PythonOperator(
        task_id="extract_rss",
        python_callable=extract_rss
    )
    
    extract_reddit = PythonOperator(
        task_id="extract_reddit",
        python_callable=extract_reddit
    )
    
    extract_hackernews = PythonOperator(
        task_id="extract_hackernews",
        python_callable=extract_hackernews
    )
    
    extract_scraper_bs4 = PythonOperator(...)
    extract_scraper_selenium = PythonOperator(...)
    extract_scraper_playwright = PythonOperator(...)
    
    # Fusion après les extractions
    merge_raw = PythonOperator(
        task_id="merge_raw",
        python_callable=merge_raw_data
    )
    
    # Dépendances: extractions >> merge
    [extract_rss, extract_reddit, extract_hackernews, 
     extract_scraper_bs4, extract_scraper_selenium, 
     extract_scraper_playwright] >> merge_raw
```

## 4.4 Fusion et transformation

### Tâche de fusion

```python
def merge_raw_data(ti):
    """Fusionne les données brutes de toutes les sources."""
    # Récupérer les fichiers depuis XCom
    rss_file = ti.xcom_pull(
        task_ids="extraction_group.extract_rss", 
        key="rss_file"
    )
    reddit_file = ti.xcom_pull(
        task_ids="extraction_group.extract_reddit", 
        key="reddit_file"
    )
    hackernews_file = ti.xcom_pull(...)
    bs4_file = ti.xcom_pull(...)
    selenium_file = ti.xcom_pull(...)
    playwright_file = ti.xcom_pull(...)
    
    all_articles = []
    
    # Fusionner tous les fichiers
    for filepath in [rss_file, reddit_file, hackernews_file, 
                      bs4_file, selenium_file, playwright_file]:
        if filepath and os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                articles = json.load(f)
                all_articles.extend(articles)
    
    # Sauvegarder le fichier fusionné
    merged_file = f"{DATA_RAW}/merged_raw_{timestamp}.json"
    with open(merged_file, "w", encoding="utf-8") as f:
        json.dump(all_articles, f, ensure_ascii=False)
    
    ti.xcom_push(key="merged_file", value=merged_file)
    ti.xcom_push(key="total_count", value=len(all_articles))
    
    return merged_file
```

### Tâche de transformation

```python
def transform_data(ti):
    """Tâche de transformation/ETL."""
    from src.transformation.pipeline import TransformationPipeline
    
    merged_file = ti.xcom_pull(
        task_ids="extraction_group.merge_raw", 
        key="merged_file"
    )
    
    pipeline = TransformationPipeline(
        input_dir=DATA_RAW,
        output_dir=DATA_PROCESSED
    )
    
    articles = pipeline.load_raw_data(os.path.basename(merged_file))
    transformed = pipeline.transform(articles)
    output_file = pipeline.save(transformed)
    
    validation = DataValidator.validate_batch(transformed)
    
    ti.xcom_push(key="transformed_file", value=output_file)
    ti.xcom_push(key="validation", value=validation)
    
    return output_file
```

## 4.5 Calcul des KPIs

```python
def calculate_kpis(ti):
    """Calcule les KPIs."""
    # Récupérer toutes les métriques
    rss_count = ti.xcom_pull(
        task_ids="extraction_group.extract_rss", 
        key="rss_count"
    ) or 0
    reddit_count = ti.xcom_pull(
        task_ids="extraction_group.extract_reddit", 
        key="reddit_count"
    ) or 0
    hackernews_count = ti.xcom_pull(
        task_ids="extraction_group.extract_hackernews", 
        key="hackernews_count"
    ) or 0
    # ... autres sources
    total_count = ti.xcom_pull(
        task_ids="extraction_group.merge_raw", 
        key="total_count"
    ) or 0
    validation = ti.xcom_pull(
        task_ids="transform", 
        key="validation"
    ) or {}
    
    # Construire le dictionnaire KPIs
    kpis = {
        "extraction_rss": rss_count,
        "extraction_reddit": reddit_count,
        "extraction_hackernews": hackernews_count,
        "total_raw": total_count,
        "valid_after_clean": validation.get("valid", 0),
        "multimodal": validation.get("multimodal", 0),
        "text_only": validation.get("text_only", 0),
        "success_rate": validation.get("valid", 0) / max(total_count, 1) * 100,
        "execution_time_seconds": exec_time,
        "timestamp": datetime.now().isoformat()
    }
    
    # Sauvegarder en JSON
    kpi_file = f"{DATA_PROCESSED}/kpis.json"
    with open(kpi_file, "w") as f:
        json.dump(kpis, f, indent=2)
    
    return kpis
```

## 4.6 Flux complet du DAG

```python
# Définir le flux des tâches
start >> extraction >> transform >> load >> kpis
```

### Schéma du DAG

```
                    ┌──────────┐
                    │  START   │
                    └────┬─────┘
                         │
                         ▼
              ┌──────────────────────┐
              │  EXTRACTION GROUP    │
              │  (TaskGroup)          │
              │                      │
              │  ┌─────┐  ┌─────┐    │
              │  │ RSS │  │Reddit│   │
              │  └─┬─┘  └─┬─┘    │
              │    │      │       │
              │  ┌─────┐  ┌─────┐    │
              │  │ HN  │  │ BS4 │    │
              │  └──┬──┘  └──┬──┘    │
              │     │      │       │
              │  ┌───────┐ ┌─────┐    │
              │  │Selenium│ │Playwright│
              │  └───┬───┘ └───┘    │
              │      │      │       │
              │      └──────┬──────┘
              │             │
              │      ┌─────┴─────┐   │
              │      │   MERGE   │   │
              │      └─────┬─────┘   │
              └────────────┼──────────┘
                           │
                           ▼
                   ┌──────────────┐
                   │  TRANSFORM   │
                   │  (ETL Pipeline)│
                   └──────┬───────┘
                          │
                          ▼
                   ┌──────────────┐
                   │    LOAD      │
                   │  (SQLite DB) │
                   └──────┬───────┘
                          │
                          ▼
                   ┌──────────────┐
                   │     KPIs     │
                   │  (Metrics)   │
                   └──────────────┘
```

## 4.7 Conclusion Mission 4 - Livrable 4

### ✅ Flux ETL orchestré (DAG Airflow)

**Fichier :** `dags/fakenews_pipeline.py`

### Tâches du DAG

| # | Tâche | Description | Type |
|---|-------|-------------|------|
| 1 | start | Initialise le timestamp | Python |
| 2 | extract_rss | Extraction 6 flux RSS | Python |
| 3 | extract_reddit | Posts Reddit | Python |
| 4 | extract_hackernews | Top stories HN | Python |
| 5 | extract_scraper_bs4 | Scraping BS4 | Python |
| 6 | extract_scraper_selenium | Scraping Selenium | Python |
| 7 | extract_scraper_playwright | Scraping Playwright | Python |
| 8 | merge_raw | Fusion des données | Python |
| 9 | transform | Pipeline ETL | Python |
| 10 | load | Insertion SQLite | Python |
| 11 | calculate_kpis | Métriques | Python |

---

### 🔑 Points clés de l'orchestration :

1. **TaskGroup** : Regroupe les extractions parallèles
2. **XCom** : Communication entre tâches
3. **Retry** : 2 tentatives avec délai de 5 minutes
4. **Schedule** : Toutes les 2 heures (`0 */2 * * *`)

---

# MISSION 5 : Monitoring & Qualité

---

## 5.1 Objectif de la mission

Créer un dashboard de suivi pour visualiser les KPIs et surveiller la qualité des données.

### Outil : Streamlit

Streamlit est un framework Python pour créer des applications de données rapidement.

```python
import streamlit as st

# Configuration de la page
st.set_page_config(
    page_title="CheckIt.AI - Dashboard",
    page_icon="📰",
    layout="wide"
)

# Titre principal
st.title("📰 CheckIt.AI - Fake News Killer")
st.markdown("**Pipeline d'extraction de données multimodales**")
```

## 5.2 Vue KPIs - Métriques clés

```python
def load_kpis():
    """Charge les KPIs depuis le fichier JSON."""
    kpi_file = DATA_DIR / "kpis.json"
    
    if not kpi_file.exists():
        return {
            "extraction_rss": 0,
            "extraction_reddit": 0,
            "total_raw": 0,
            "valid_after_clean": 0,
            "multimodal": 0,
        }
    
    with open(kpi_file, "r") as f:
        return json.load(f)

def main():
    # ...
    kpis = load_kpis()
    
    # Métriques en colonnes
    col1, col2, col3, col4 = st.columns(4)
    
    with col1:
        st.metric(
            "Articles extraits (RSS)",
            kpis.get("extraction_rss", 0)
        )
    
    with col2:
        st.metric(
            "Posts Reddit",
            kpis.get("extraction_reddit", 0)
        )
    
    with col3:
        st.metric(
            "Articles multimodaux",
            kpis.get("multimodal", 0)
        )
    
    with col4:
        st.metric(
            "Taux de conservation",
            f"{kpis.get('success_rate', 0):.1f}%"
        )
```

### Graphiques

```python
# Répartition des données
chart_data = pd.DataFrame({
    "Catégorie": ["Multimodaux", "Texte seul", "Filtrés"],
    "Nombre": [
        kpis.get("multimodal", 0),
        kpis.get("text_only", 0),
        kpis.get("total_raw", 0) - kpis.get("valid_after_clean", 0)
    ]
})
st.bar_chart(chart_data.set_index("Catégorie"))
```

## 5.3 Vue Données brutes

```python
def load_raw_data():
    """Charge les données brutes du dernier run."""
    merged_files = list(RAW_DIR.glob("merged_raw_*.json"))
    
    if not merged_files:
        return pd.DataFrame()
    
    latest = max(merged_files, key=os.path.getmtime)
    
    with open(latest, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    return pd.DataFrame(data)

# Page Données brutes
elif page == "Données brutes":
    st.header("📥 Données brutes extraites")
    
    df_raw = load_raw_data()
    
    if df_raw.empty:
        st.warning("Aucune donnée brute disponible.")
    else:
        st.write(f"**Total: {len(df_raw)} articles**")
        
        # Filtres
        col1, col2 = st.columns(2)
        with col1:
            type_filter = st.multiselect(
                "Filtrer par type",
                options=df_raw["type"].unique()
            )
        with col2:
            source_filter = st.multiselect(
                "Filtrer par source",
                options=df_raw["source"].unique()
            )
        
        # Affichage du tableau
        st.dataframe(
            df_raw[["title", "source", "type", "published_at"]].head(200)
        )
```

## 5.4 Vue Classification

```python
# Tableau classification
st.subheader("Classification des articles")

if not df_transformed.empty and "classification" in df_transformed.columns:
    # Extraire la colonne classification
    df_class = df_transformed["classification"].apply(pd.Series)
    class_counts = df_class["category"].value_counts().reset_index()
    class_counts.columns = ["Catégorie", "Nombre"]
    
    # Afficher les stats
    st.dataframe(class_counts)
    
    # Statistiques nettoyés vs bruts
    st.subheader("Articles nettoyés")
    bruts = int(kpis.get('total_raw', 0))
    valides = int(kpis.get('valid_after_clean', 0))
    rejetes = bruts - valides
    st.write(f"Bruts: {bruts} → Valides: {valides} (rejetés: {rejetes})")
    
    # Stats par source
    source_stats = df_transformed.groupby(["source", "type"]).size().reset_index(name="Nombre")
    st.dataframe(source_stats)
```

### Exemple d'affichage

```
Classification des articles
┌──────────────────┬──────────┐
│ Catégorie        │ Nombre   │
├──────────────────┼──────────┤
│ opinion          │ 45       │
│ neutre           │ 28       │
│ indetermine     │ 12       │
│ desinformation  │ 5        │
└──────────────────┴──────────┘
```

## 5.5 Vue Monitoring

```python
elif page == "Monitoring":
    st.header("🔍 Monitoring du pipeline")
    
    # État du système
    st.subheader("État des composants")
    
    # Vérification des fichiers
    raw_files = len(list(RAW_DIR.glob("*.json")))
    processed_files = len(list(DATA_DIR.glob("*.json")))
    
    col1, col2 = st.columns(2)
    
    with col1:
        status = "✅" if raw_files > 0 else "❌"
        st.write(f"{status} Données brutes: {raw_files} fichiers")
    
    with col2:
        status = "✅" if processed_files > 0 else "❌"
        st.write(f"{status} Données traitées: {processed_files} fichiers")
    
    # Logs
    st.subheader("Derniers logs")
    
    if kpis.get("timestamp"):
        st.code(f"""
Timestamp: {kpis['timestamp']}
RSS: {kpis.get('extraction_rss', 0)} articles
Reddit: {kpis.get('extraction_reddit', 0)} articles
Total brut: {kpis.get('total_raw', 0)}
Valides: {kpis.get('valid_after_clean', 0)}
Taux réussite: {kpis.get('success_rate', 0):.1f}%
""", language="text")
    
    # Plan de monitoring
    st.subheader("📋 Plan de monitoring")
    
    monitoring_plan = """
| Métrique | Objectif | Alerte si |
|----------|----------|-----------|
| Temps d'exécution | < 5 min | > 10 min |
| Articles extraits | > 50 | < 20 |
| Taux de validité | > 80% | < 50% |
| Articles multimodaux | > 30% | < 10% |
"""
    st.markdown(monitoring_plan)
```

## 5.6 Lancement du Dashboard

### Commande

```bash
# Lancer Streamlit
pixi run streamlit run dashboard/app.py
```

### Interface

```
┌─────────────────────────────────────────────────────────────┐
│  📰 CheckIt.AI - Fake News Killer                          │
│                                                             │
│  ┌──────────────┐                                          │
│  │ Navigation   │  ┌────────────────────────────────────┐  │
│  │              │  │                                    │  │
│  │ ○ KPIs       │  │   [Contenu selon la page]         │  │
│  │ ○ Données    │  │                                    │  │
│    brutes      │  │   • Métriques                      │  │
│  ○ Données     │  │   • Graphiques                     │  │
│    transformées│  │   • Tableaux de données            │  │
│  ○ Monitoring  │  │                                    │  │
│              │  │                                    │  │
│  ─────────────│  │                                    │  │
│  P12 - IA    │  └────────────────────────────────────┘  │
└──────────────┘                                          │
```

### Navigation

| Page | Description |
|------|-------------|
| **KPIs** | Métriques clés et graphiques |
| **Données brutes** | Articles extraits (non transformés) |
| **Données transformées** | Articles nettoyés et classifiés |
| **Monitoring** | État du système et plan de suivi |

## 5.7 Conclusion Mission 5 - Livrable 5

### ✅ Dashboard KPI et plan de monitoring

**Fichier :** `dashboard/app.py`

### Vues disponibles

| Vue | Contenu | Métriques |
|-----|---------|-----------|
| **KPIs** | Métriques clés | RSS, Reddit, Multimodaux, Taux |
| **Données brutes** | Articles extraits | Filtrables par type/source |
| **Données transformées** | Articles classifiés | Classification Opinion/Disinfo |
| **Monitoring** | État du système | Fichiers, Logs, Plan |

---

### 🔑 Points clés du monitoring :

1. **4 vues distinctes** : KPIs, Brutes, Transformées, Monitoring
2. **Filtres interactifs** : Par type et source
3. **Graphiques temps réel** : Évolution des métriques
4. **Plan de monitoring** : Objectifs et alertes définis

---

# RÉSUMÉ FINAL - Les 5 Livrables

---

| # | Livrable | Fichier | Status |
|---|----------|---------|--------|
| 1 | Rapport d'exploration | `docs/exploration_sources.md` | ✅ |
| 2 | Scripts d'extraction | `src/extraction/*.py` | ✅ |
| 3 | Pipeline ETL + MCD | `src/transformation/` + `src/models/schema.mmd` | ✅ |
| 4 | DAG Airflow | `dags/fakenews_pipeline.py` | ✅ |
| 5 | Dashboard Streamlit | `dashboard/app.py` | ✅ |

---

## 🏗️ Architecture complète

```
┌─────────────────────────────────────────────────────────────────┐
│                    CHECKIT.AI - FAKE NEWS KILLER                │
│                          Projet P12 - Formation IA              │
└─────────────────────────────────────────────────────────────────┘

    EXTRACTION                TRANSFORMATION           MONITORING
 ┌────────────────┐      ┌────────────────┐     ┌────────────────┐
 │ • RSS Fetcher  │      │ • DataCleaner  │     │ • Streamlit    │
 │ • API Clients  │ ──►  │ • Normalizer   │ ──► │ • KPIs Dashboard│
 │ • BeautifulSoup│      │ • Classifier   │     │ • Plan monitoring│
 │ • Selenium     │      │ • Validator    │     └────────────────┘
 │ • Playwright   │      │ • MCD Mermaid  │
 └───────┬────────┘      └───────┬────────┘
         │                       │
         ▼                       ▼
 ┌───────────────────────────────────────────┐
 │            ORCHESTRATION AIRFLOW          │
 │                                           │
 │  start → extraction → merge → transform → load → KPIs  │
 │                  (parallèle)                           │
 └───────────────────────────────────────────┘
                          │
                          ▼
                   ┌─────────────┐
                   │   SQLite    │
                   │  fakenews.db│
                   └─────────────┘
```

---

## 📊 Compétences acquises

| Domaine | Technologies | Tags |
|---------|--------------|------|
| **Data Ingestion** | WebScraping, APIs REST, RSS | #WebScraping #API_Rest #Multimodal |
| **Data Transformation** | ETL, Cleaning, Normalization | #ETL #DataCleaning #Mermaid |
| **Orchestration** | Apache Airflow, DAGs | #ApacheAirflow #DAG #Scheduling |
| **Operations** | KPIs, Monitoring, Logging | #KPI #DataMonitoring #ErrorHandling |

## 🚀 Lancement du projet

### Prérequis

```bash
# Installer Pixi (gestionnaire de paquets)
curl -fsSL https://pixi.sh/install.sh | bash
```

### Commandes principales

```bash
# 1. Extraction manuelle
pixi run python src/extraction/main.py --source all

# 2. Transformation des données
pixi run python src/transformation/pipeline.py --file merged_raw_*.json

# 3. Lancement du dashboard
pixi run streamlit run dashboard/app.py

# 4. Lancement d'Airflow
./run_airflow.sh
```

---

### Déploiement Kubernetes

```yaml
# k8s/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: checkit-ai-dashboard
  namespace: sophia-apps
spec:
  replicas: 1
  selector:
    matchLabels:
      app: checkit-ai
  template:
    spec:
      containers:
      - name: dashboard
        image: checkit-ai:latest
        ports:
        - containerPort: 8501
```

---

## ✅ Projet CheckIt.AI - Fake News Killer

**Auteur :** Damien G.
**Formation :** Projet P12 - Formation IA
**Date :** Avril 2026

🎯 *Toutes les missions accomplies, tous les livrables produits.*